<a href="https://colab.research.google.com/github/govardhansnair/binomial-option-pricing-framework/blob/main/Binomial_Project_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:


# 1. BINOMIAL OPTION PRICING FUNCTION

import math


def price_option(spot, strike, rate, volatility, maturity,
                 steps=200, option_type="call",
                 exercise="european", dividend_yield=0.0,
                 return_tree=False):

    # Validate inputs
    if spot <= 0 or strike <= 0 or volatility <= 0 or maturity <= 0 or steps < 1:
        raise ValueError("Invalid input values")

    if option_type not in ("call", "put"):
        raise ValueError("Option type must be call or put")

    if exercise not in ("european", "american"):
        raise ValueError("Exercise must be european or american")

    # Calculate time interval
    dt = maturity / steps

    # Calculate CRR upward and downward factors
    u = math.exp(volatility * math.sqrt(dt))
    d = 1 / u

    # Calculate risk-neutral probability
    q = (math.exp((rate - dividend_yield) * dt) - d) / (u - d)

    if not 0 <= q <= 1:
        raise ValueError("Invalid risk-neutral probability")

    # Calculate discount factor
    discount = math.exp(-rate * dt)


    # 2. GENERATE STOCK PRICE TREE


    stock_tree = []

    for i in range(steps + 1):

        row = []

        for j in range(i + 1):

            stock_price = spot * (u ** j) * (d ** (i - j))

            row.append(stock_price)

        stock_tree.append(row)


    # 3. CALCULATE TERMINAL OPTION PAYOFFS


    last_prices = stock_tree[-1]

    if option_type == "call":

        values = [max(s - strike, 0) for s in last_prices]

    else:

        values = [max(strike - s, 0) for s in last_prices]

    # Create option tree
    option_tree = [None] * (steps + 1)

    option_tree[steps] = values.copy()


    # 4. BACKWARD INDUCTION

    for i in range(steps - 1, -1, -1):

        new_values = []

        for j in range(i + 1):

            # Risk-neutral continuation value
            continuation = discount*(q*values[j+1]+(1-q)*values[j])

            # Check for American early exercise
            if exercise == "american":

                stock = stock_tree[i][j]

                if option_type == "call":

                    intrinsic = max(stock - strike, 0)

                else:

                    intrinsic = max(strike - stock, 0)

                value = max(intrinsic, continuation)

            else:

                value = continuation

            new_values.append(value)

        values = new_values

        option_tree[i] = values.copy()


    # 5. CALCULATE DELTA


    delta = (option_tree[1][1]-option_tree[1][0])/(stock_tree[1][1]-stock_tree[1][0])

    # Return full tree if requested
    if return_tree:

        return {
            "price": values[0],
            "delta": delta,
            "stock_tree": stock_tree,
            "option_tree": option_tree,
            "up_factor": u,
            "down_factor": d,
            "risk_neutral_probability": q,
            "dt": dt
        }

    return values[0]



# 6. CALCULATE OPTION GREEKS


def calculate_greeks(spot, strike, rate, volatility, maturity,
                     steps=200, option_type="call",
                     exercise="european", dividend_yield=0.0):

    # Calculate original option price
    base_price = price_option(spot, strike, rate, volatility, maturity,
                 steps, option_type, exercise, dividend_yield)

    # Small changes for numerical calculations
    h = 0.01
    spot_change = spot * 0.001

    # Calculate prices at different stock prices
    price_up = price_option(spot + spot_change, strike, rate, volatility,
        maturity, steps, option_type, exercise, dividend_yield)

    price_down = price_option(spot - spot_change, strike, rate, volatility,
        maturity, steps, option_type, exercise, dividend_yield)

    # Delta
    delta = (price_up - price_down) / (2 * spot_change)

    # Gamma
    gamma = (price_up - 2 * base_price + price_down) / (spot_change ** 2)

    # Calculate prices at different volatility levels
    volatility_up = price_option(
        spot, strike, rate, volatility + h,
        maturity, steps, option_type, exercise, dividend_yield
    )

    volatility_down = price_option(
        spot, strike, rate, volatility - h,
        maturity, steps, option_type, exercise, dividend_yield
    )

    # Vega
    vega = (volatility_up - volatility_down) / (2 * h)

    # Theta
    time_change = 1 / 365

    shorter_price = price_option(
        spot, strike, rate, volatility,
        maturity - time_change, steps,
        option_type, exercise, dividend_yield
    )

    theta = (shorter_price - base_price)

    return {
        "price": base_price,
        "delta": delta,
        "gamma": gamma,
        "vega": vega,
        "vega_per_1_percent": vega * 0.01,
        "theta_per_day": theta
    }



# 7. RUN THE MODEL


if __name__ == "__main__":

    S0 = 100
    K = 100
    r = 0.05
    sigma = 0.20
    T = 1
    N = 200

    # Calculate European call
    call_price = price_option(
        S0, K, r, sigma, T, N,
        option_type="call"
    )

    # Calculate European put
    put_price = price_option(
        S0, K, r, sigma, T, N,
        option_type="put"
    )

    # Calculate American put
    american_put = price_option(
        S0, K, r, sigma, T, N,
        option_type="put",
        exercise="american"
    )

    print("EUROPEAN CALL:", round(call_price, 4))
    print("EUROPEAN PUT:", round(put_price, 4))
    print("AMERICAN PUT:", round(american_put, 4))

    # Calculate Greeks
    greeks = calculate_greeks(
        S0, K, r, sigma, T, N
    )

    print("\nOPTION GREEKS")

    for name, value in greeks.items():

        print(name, ":", round(value, 6))

EUROPEAN CALL: 10.4406
EUROPEAN PUT: 5.5635
AMERICAN PUT: 6.0864

OPTION GREEKS
price : 10.440591
delta : 0.636671
gamma : 0.530005
vega : 37.473948
vega_per_1_percent : 0.374739
theta_per_day : -0.017567
